# Fundus preprocessing, FOV, and quality audit

**Public role:** Core preprocessing-quality notebook.

**Relation to the manuscript:** Supports the preprocessing-derived quality score and FOV checks.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `03_fundus_preprocessing_fov_quality_audit.ipynb`


In [ ]:
import os
# Cell 1: Load split metadata and project paths

from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
CONFIG_PATH = JOURNAL_ROOT / "configs" / "project_config.json"
PATH_REGISTRY_PATH = JOURNAL_ROOT / "configs" / "path_registry.csv"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing config: {CONFIG_PATH}")

if not PATH_REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Missing path registry: {PATH_REGISTRY_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

path_registry = pd.read_csv(PATH_REGISTRY_PATH)
PATHS = {
    row["path_key"]: Path(row["path"])
    for _, row in path_registry.iterrows()
}

IMAGE_SPLIT_CSV = PATHS["splits"] / "train_val_test_image_split.csv"
SPLIT_CONFIG_JSON = PATHS["splits"] / "split_config.json"

if not IMAGE_SPLIT_CSV.exists():
    raise FileNotFoundError(f"Missing image split file: {IMAGE_SPLIT_CSV}")

if not SPLIT_CONFIG_JSON.exists():
    raise FileNotFoundError(f"Missing split config: {SPLIT_CONFIG_JSON}")

metadata = pd.read_csv(IMAGE_SPLIT_CSV)

required_columns = [
    "image_id",
    "image_name",
    "patient_id",
    "label",
    "label_raw",
    "quality_score",
    "image_path",
    "split",
]

missing_columns = [col for col in required_columns if col not in metadata.columns]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {list(metadata.columns)}"
    )

metadata["image_path"] = metadata["image_path"].astype(str)
metadata["path_exists"] = metadata["image_path"].apply(lambda x: Path(x).exists())

missing_paths = int((~metadata["path_exists"]).sum())
split_counts = metadata["split"].value_counts().to_dict()
label_counts = metadata["label_raw"].value_counts().to_dict()

input_audit = pd.DataFrame(
    [
        {"check": "n_images", "value": len(metadata), "expected": PROJECT_CONFIG["expected_hygd_images"], "passed": len(metadata) == PROJECT_CONFIG["expected_hygd_images"]},
        {"check": "n_patients", "value": metadata["patient_id"].nunique(), "expected": PROJECT_CONFIG["expected_hygd_patients"], "passed": metadata["patient_id"].nunique() == PROJECT_CONFIG["expected_hygd_patients"]},
        {"check": "missing_image_paths", "value": missing_paths, "expected": 0, "passed": missing_paths == 0},
        {"check": "split_values", "value": ",".join(sorted(metadata["split"].unique())), "expected": "test,train,val", "passed": sorted(metadata["split"].unique()) == ["test", "train", "val"]},
    ]
)

input_audit_csv = PATHS["results_03_preprocessing_quality"] / "notebook03_input_audit.csv"
input_audit.to_csv(input_audit_csv, index=False)

failed_checks = input_audit.loc[~input_audit["passed"], "check"].tolist()

print(f"Images: {len(metadata)}")
print(f"Patients: {metadata['patient_id'].nunique()}")
print(f"Splits: {split_counts}")
print(f"Labels: {label_counts}")
print(f"Missing image paths: {missing_paths}")
print(f"Input audit: {input_audit_csv}")

if failed_checks:
    raise RuntimeError(f"Failed input checks: {failed_checks}")

print("Status: PASS")

In [ ]:
# Cell 2: Image readability and geometry audit

image_audit_rows = []

for _, row in metadata.iterrows():
    image_path = Path(row["image_path"])
    img = cv2.imread(str(image_path), cv2.IMREAD_UNCHANGED)

    if img is None:
        image_audit_rows.append(
            {
                "image_id": row["image_id"],
                "image_name": row["image_name"],
                "image_path": str(image_path),
                "readable": False,
                "height": np.nan,
                "width": np.nan,
                "channels": np.nan,
                "dtype": "",
                "file_size_mb": round(image_path.stat().st_size / (1024 ** 2), 6) if image_path.exists() else np.nan,
            }
        )
        continue

    if img.ndim == 2:
        height, width = img.shape
        channels = 1
    else:
        height, width, channels = img.shape

    image_audit_rows.append(
        {
            "image_id": row["image_id"],
            "image_name": row["image_name"],
            "image_path": str(image_path),
            "readable": True,
            "height": int(height),
            "width": int(width),
            "channels": int(channels),
            "dtype": str(img.dtype),
            "file_size_mb": round(image_path.stat().st_size / (1024 ** 2), 6),
        }
    )

image_geometry_audit = pd.DataFrame(image_audit_rows)

unreadable_count = int((~image_geometry_audit["readable"]).sum())
unique_shapes = (
    image_geometry_audit[image_geometry_audit["readable"]]
    .groupby(["height", "width", "channels"])
    .size()
    .reset_index(name="n_images")
    .sort_values("n_images", ascending=False)
)

image_geometry_audit_csv = PATHS["results_03_preprocessing_quality"] / "notebook03_image_geometry_audit.csv"
unique_shapes_csv = PATHS["results_03_preprocessing_quality"] / "notebook03_unique_image_shapes.csv"

image_geometry_audit.to_csv(image_geometry_audit_csv, index=False)
unique_shapes.to_csv(unique_shapes_csv, index=False)

print(f"Readable images: {len(image_geometry_audit) - unreadable_count}/{len(image_geometry_audit)}")
print(f"Unreadable images: {unreadable_count}")
print(f"Unique shapes: {len(unique_shapes)}")
print(f"Most common shape: {unique_shapes.iloc[0].to_dict()}")
print(f"Geometry audit: {image_geometry_audit_csv}")

if unreadable_count > 0:
    unreadable = image_geometry_audit.loc[~image_geometry_audit["readable"], "image_name"].tolist()
    raise RuntimeError(f"Unreadable images found: {unreadable[:20]}")

print("Status: PASS")

In [ ]:
# Cell 3: Generate FOV masks and image quality features

FOV_SIZE = 512
FOV_MASK_DIR = PATHS["fov_masks"] / "masks_512"
FOV_MASK_DIR.mkdir(parents=True, exist_ok=True)

def largest_component(mask):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    if num_labels <= 1:
        return mask.astype(np.uint8)

    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == largest_label).astype(np.uint8)

def build_fov_mask_512(image_bgr):
    image_512 = cv2.resize(image_bgr, (FOV_SIZE, FOV_SIZE), interpolation=cv2.INTER_AREA)
    hsv = cv2.cvtColor(image_512, cv2.COLOR_BGR2HSV)

    saturation = hsv[:, :, 1]
    value = hsv[:, :, 2]

    mask = ((value > 18) & (saturation > 8)).astype(np.uint8)
    mask = cv2.medianBlur(mask * 255, 5)
    mask = (mask > 0).astype(np.uint8)

    kernel = np.ones((9, 9), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = largest_component(mask)

    return image_512, mask

fov_rows = []

for _, row in metadata.iterrows():
    image_path = Path(row["image_path"])
    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

    if image_bgr is None:
        raise RuntimeError(f"Unreadable image: {image_path}")

    image_512, fov_mask = build_fov_mask_512(image_bgr)

    gray_512 = cv2.cvtColor(image_512, cv2.COLOR_BGR2GRAY)
    fov_pixels = gray_512[fov_mask > 0]

    ys, xs = np.where(fov_mask > 0)
    if len(xs) == 0:
        raise RuntimeError(f"Empty FOV mask: {row['image_name']}")

    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())

    mask_path = FOV_MASK_DIR / f"{row['image_id']}_fov_512.png"
    cv2.imwrite(str(mask_path), fov_mask * 255)

    fov_rows.append(
        {
            "image_id": row["image_id"],
            "image_name": row["image_name"],
            "patient_id": row["patient_id"],
            "split": row["split"],
            "label": int(row["label"]),
            "quality_score": float(row["quality_score"]),
            "fov_mask_path": str(mask_path),
            "fov_area_ratio": float(fov_mask.mean()),
            "fov_bbox_x1": x1,
            "fov_bbox_y1": y1,
            "fov_bbox_x2": x2,
            "fov_bbox_y2": y2,
            "fov_mean_intensity": float(np.mean(fov_pixels)),
            "fov_std_intensity": float(np.std(fov_pixels)),
            "laplacian_var_512": float(cv2.Laplacian(gray_512, cv2.CV_64F).var()),
        }
    )

fov_manifest = pd.DataFrame(fov_rows)

fov_manifest_csv = PATHS["project_data"] / "fov_masks" / "fov_manifest_512.csv"
fov_manifest_results_csv = PATHS["results_03_preprocessing_quality"] / "notebook03_fov_manifest_512.csv"

fov_manifest.to_csv(fov_manifest_csv, index=False)
fov_manifest.to_csv(fov_manifest_results_csv, index=False)

mask_count = len(list(FOV_MASK_DIR.glob("*.png")))
empty_masks = int((fov_manifest["fov_area_ratio"] <= 0).sum())

print(f"FOV masks: {mask_count}")
print(f"FOV area ratio: mean={fov_manifest['fov_area_ratio'].mean():.3f}, std={fov_manifest['fov_area_ratio'].std():.3f}")
print(f"Empty masks: {empty_masks}")
print(f"FOV manifest: {fov_manifest_csv}")

if mask_count != len(metadata):
    raise RuntimeError(f"Expected {len(metadata)} FOV masks, found {mask_count}")

if empty_masks > 0:
    raise RuntimeError("Empty FOV masks found.")

print("Status: PASS")

In [ ]:
# Cell 4: Build preprocessing and quality manifest

quality_labels = ["low_quality", "medium_quality", "high_quality"]

preprocessing_manifest = metadata.merge(
    image_geometry_audit.drop(columns=["image_path"]),
    on=["image_id", "image_name"],
    how="left"
)

preprocessing_manifest = preprocessing_manifest.merge(
    fov_manifest.drop(columns=["patient_id", "split", "label", "quality_score"]),
    on=["image_id", "image_name"],
    how="left"
)

try:
    preprocessing_manifest["quality_tertile"] = pd.qcut(
        preprocessing_manifest["quality_score"],
        q=3,
        labels=quality_labels,
    ).astype(str)
    quality_tertile_method = "qcut_quality_score"
except ValueError:
    preprocessing_manifest["quality_tertile"] = pd.qcut(
        preprocessing_manifest["quality_score"].rank(method="first"),
        q=3,
        labels=quality_labels,
    ).astype(str)
    quality_tertile_method = "qcut_ranked_quality_score"

quality_tertile_summary = (
    preprocessing_manifest.groupby("quality_tertile")
    .agg(
        n_images=("image_id", "count"),
        n_patients=("patient_id", "nunique"),
        n_positive=("label", "sum"),
        n_negative=("label", lambda x: int((x == 0).sum())),
        quality_min=("quality_score", "min"),
        quality_max=("quality_score", "max"),
        quality_mean=("quality_score", "mean"),
        fov_area_mean=("fov_area_ratio", "mean"),
        sharpness_mean=("laplacian_var_512", "mean"),
    )
    .reset_index()
)

split_quality_summary = (
    preprocessing_manifest.groupby(["split", "quality_tertile"])
    .agg(
        n_images=("image_id", "count"),
        n_patients=("patient_id", "nunique"),
        quality_mean=("quality_score", "mean"),
    )
    .reset_index()
)

required_manifest_columns = [
    "image_id",
    "image_name",
    "patient_id",
    "label",
    "label_raw",
    "split",
    "quality_score",
    "quality_tertile",
    "height",
    "width",
    "channels",
    "fov_mask_path",
    "fov_area_ratio",
    "fov_mean_intensity",
    "fov_std_intensity",
    "laplacian_var_512",
]

missing_manifest_columns = [
    col for col in required_manifest_columns
    if col not in preprocessing_manifest.columns
]

if missing_manifest_columns:
    raise ValueError(f"Missing manifest columns: {missing_manifest_columns}")

missing_values = preprocessing_manifest[required_manifest_columns].isna().sum()
columns_with_missing = missing_values[missing_values > 0].to_dict()

preprocessing_manifest_csv = PATHS["preprocessed"] / "preprocessing_quality_manifest.csv"
quality_tertile_summary_csv = PATHS["results_03_preprocessing_quality"] / "notebook03_quality_tertile_summary.csv"
split_quality_summary_csv = PATHS["results_03_preprocessing_quality"] / "notebook03_split_quality_summary.csv"

preprocessing_manifest.to_csv(preprocessing_manifest_csv, index=False)
quality_tertile_summary.to_csv(quality_tertile_summary_csv, index=False)
split_quality_summary.to_csv(split_quality_summary_csv, index=False)

print(f"Manifest rows: {len(preprocessing_manifest)}")
print(f"Quality tertile method: {quality_tertile_method}")
print(f"Quality tertiles: {preprocessing_manifest['quality_tertile'].value_counts().to_dict()}")
print(f"Columns with missing values: {columns_with_missing}")
print(f"Preprocessing manifest: {preprocessing_manifest_csv}")

if len(preprocessing_manifest) != len(metadata):
    raise RuntimeError("Unexpected preprocessing manifest row count.")

if columns_with_missing:
    raise RuntimeError(f"Missing values found: {columns_with_missing}")

print("Status: PASS")

In [ ]:
# Cell 5: Final status

required_outputs = {
    "input_audit_csv": input_audit_csv,
    "image_geometry_audit_csv": image_geometry_audit_csv,
    "unique_shapes_csv": unique_shapes_csv,
    "fov_manifest_csv": fov_manifest_csv,
    "preprocessing_manifest_csv": preprocessing_manifest_csv,
    "quality_tertile_summary_csv": quality_tertile_summary_csv,
    "split_quality_summary_csv": split_quality_summary_csv,
}

output_checks = {
    key: path.exists()
    for key, path in required_outputs.items()
}

content_checks = {
    "n_images": len(preprocessing_manifest) == int(PROJECT_CONFIG["expected_hygd_images"]),
    "all_images_readable": int((~image_geometry_audit["readable"]).sum()) == 0,
    "all_fov_masks_created": len(list(FOV_MASK_DIR.glob("*.png"))) == len(preprocessing_manifest),
    "no_empty_fov_masks": int((preprocessing_manifest["fov_area_ratio"] <= 0).sum()) == 0,
    "no_missing_quality_tertiles": int(preprocessing_manifest["quality_tertile"].isna().sum()) == 0,
    "valid_splits": sorted(preprocessing_manifest["split"].unique()) == ["test", "train", "val"],
}

final_checks = {**output_checks, **content_checks}
failed_checks = [key for key, passed in final_checks.items() if not passed]
status = "PASS" if not failed_checks else "FAIL"

notebook03_status = {
    "notebook": "03_fundus_preprocessing_fov_quality_audit",
    "status": status,
    "n_images": int(len(preprocessing_manifest)),
    "n_unique_shapes": int(len(unique_shapes)),
    "fov_masks": int(len(list(FOV_MASK_DIR.glob("*.png")))),
    "fov_area_ratio_mean": float(preprocessing_manifest["fov_area_ratio"].mean()),
    "fov_area_ratio_std": float(preprocessing_manifest["fov_area_ratio"].std()),
    "quality_tertile_counts": {
        key: int(value)
        for key, value in preprocessing_manifest["quality_tertile"].value_counts().to_dict().items()
    },
    "failed_checks": failed_checks,
    "next_notebook": "04_external_od_segmentor_validation.ipynb",
}

status_json = PATHS["results_03_preprocessing_quality"] / "notebook03_final_status.json"
status_txt = PATHS["results_03_preprocessing_quality"] / "notebook03_FINAL_STATUS.txt"
status_checks_csv = PATHS["results_03_preprocessing_quality"] / "notebook03_final_status_checks.csv"

with open(status_json, "w", encoding="utf-8") as f:
    json.dump(notebook03_status, f, indent=2)

with open(status_txt, "w", encoding="utf-8") as f:
    for key, value in notebook03_status.items():
        f.write(f"{key}: {value}\n")

pd.DataFrame(
    [{"check": key, "passed": bool(value)} for key, value in final_checks.items()]
).to_csv(status_checks_csv, index=False)

print(f"Status: {status}")
print(f"Images: {len(preprocessing_manifest)}")
print(f"FOV masks: {len(list(FOV_MASK_DIR.glob('*.png')))}")
print(f"FOV area mean: {preprocessing_manifest['fov_area_ratio'].mean():.3f}")
print(f"Quality tertiles: {preprocessing_manifest['quality_tertile'].value_counts().to_dict()}")
print(f"Final status: {status_json}")

if failed_checks:
    raise RuntimeError(f"Notebook 03 failed checks: {failed_checks}")